In [ ]:
# Setup - Works on both CPU and GPU, and doesn't require bitsandbytes
!pip install -U pip setuptools wheel
!pip install transformers accelerate huggingface_hub
!pip install PyPDF2 nltk sentence-transformers faiss-cpu torch scikit-learn
!pip install transformers sentence-transformers faiss-cpu torch PyPDF2 nltk scikit-learn huggingface_hub
!pip install PyPDF2 pdfplumber
!pip install sentence-transformers faiss-gpu pdfplumber

ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


In [ ]:
import os
import numpy as np
import faiss
import torch
import PyPDF2
import nltk
import re
from typing import List, Dict
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from huggingface_hub import login
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from google.colab import drive
drive.mount('/content/drive')

# Ensure NLTK resources are downloaded
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    print("Downloading NLTK punkt resource...")
    nltk.download('punkt', quiet=False)

Mounted at /content/drive


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:

# ✅ Use lightweight and reliable model
model_name = "google/gemma-2b-it"  # Fast, fits in Colab Free or Pro, no quantization needed
#model_name = "microsoft/phi-2"  # Faster, lighter, good quality
#model_name = "meta-llama/Llama-3.2-1B"

In [ ]:
login(token="*******")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
class ContextTracker:
    def __init__(self, max_history_length=5):
        """
        Initialize context tracker with semantic embedding
        """
        self.max_history_length = max_history_length
        self.chat_history = []

        # Semantic embedding model for context understanding
        self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

    def add_interaction(self, user_message, emotion, assistant_response):
        """
        Add interaction to chat history with semantic embedding
        """
        interaction = {
            "user_message": user_message,
            "emotion": emotion,
            "assistant_response": assistant_response,
            "embedding": self.embedding_model.encode([user_message])[0]
        }

        # Maintain maximum history length
        if len(self.chat_history) >= self.max_history_length:
            self.chat_history.pop(0)

        self.chat_history.append(interaction)

    def find_similar_context(self, current_message, threshold=0.7):
        """
        Find similar previous context using semantic similarity
        """
        if not self.chat_history:
            return None

        current_embedding = self.embedding_model.encode([current_message])[0]

        # Compare with previous interactions
        similarities = [
            cosine_similarity(
                current_embedding.reshape(1, -1),
                interaction['embedding'].reshape(1, -1)
            )[0][0]
            for interaction in self.chat_history
        ]

        max_similarity = max(similarities)
        if max_similarity >= threshold:
            return self.chat_history[similarities.index(max_similarity)]

        return None

In [ ]:
class PDFKnowledgeBaseRAG:
    def __init__(self,
                 pdf_folder_path: str,
                 embedding_model: str = 'all-MiniLM-L6-v2',
                 chunk_size: int = 300,
                 overlap: int = 50):
        """
        Initialize PDF-based RAG Knowledge Base
        """
        self.pdf_folder_path = pdf_folder_path
        self.chunk_size = chunk_size
        self.overlap = overlap

        self.embedding_model = SentenceTransformer(embedding_model)

        self.raw_documents = []
        self.text_chunks = []
        self.metadata = []

        self.load_pdf_documents()
        self.faiss_index = self._create_faiss_index()

    def _extract_text_from_pdf(self, pdf_path: str) -> str:
        """
        Extract text from a single PDF file with improved robustness
        """
        try:
            with open(pdf_path, 'rb') as file:
                # Use PyPDF2 for text extraction
                reader = PyPDF2.PdfReader(file)
                text = ''

                # Extract text from each page
                for page in reader.pages:
                    # Try multiple text extraction methods
                    page_text = page.extract_text() or ''

                    # Additional extraction if first method fails
                    if not page_text.strip():
                        try:
                            import pdfplumber
                            with pdfplumber.open(pdf_path) as pdf:
                                page_text = pdf.pages[reader.pages.index(page)].extract_text() or ''
                        except ImportError:
                            pass

                    text += page_text + '\n'

            return text
        except Exception as e:
            print(f"Error extracting text from {pdf_path}: {e}")
            return ""

    def _chunk_text(self, text: str) -> List[str]:
        """
        Chunk text into overlapping segments
        """
        # Use NLTK for sentence tokenization
        try:
            sentences = nltk.sent_tokenize(text)
        except LookupError:
            # Fallback method if punkt is not available
            sentences = text.split('. ')

        chunks = []
        current_chunk = []
        current_length = 0

        for sentence in sentences:
            # Simple length estimation (approximation)
            sentence_tokens = sentence.split()

            if current_length + len(sentence_tokens) > self.chunk_size:
                # If adding this sentence would exceed chunk size, start a new chunk
                chunks.append(' '.join(current_chunk))

                # Reset with overlap
                current_chunk = current_chunk[-self.overlap:] + [sentence]
                current_length = len(current_chunk[0].split()) + len(sentence_tokens)
            else:
                current_chunk.append(sentence)
                current_length += len(sentence_tokens)

        # Add final chunk
        if current_chunk:
            chunks.append(' '.join(current_chunk))

        return chunks

    def load_pdf_documents(self):
        """
        Load and process all PDFs in the specified folder
        """
        for filename in os.listdir(self.pdf_folder_path):
            if filename.endswith('.pdf'):
                pdf_path = os.path.join(self.pdf_folder_path, filename)

                # Extract text
                text = self._extract_text_from_pdf(pdf_path)

                # Clean text
                text = re.sub(r'\s+', ' ', text).strip()

                if text:
                    self.raw_documents.append(text)

                    # Chunk text
                    chunks = self._chunk_text(text)
                    self.text_chunks.extend(chunks)

                    # Create metadata for each chunk
                    metadata = [{'source': filename, 'chunk_idx': i} for i in range(len(chunks))]
                    self.metadata.extend(metadata)

    def _create_faiss_index(self):
        """
        Create FAISS index for semantic search
        """
        # Embed text chunks
        embeddings = self.embedding_model.encode(self.text_chunks)

        # Create FAISS index
        dimension = embeddings.shape[1]
        index = faiss.IndexFlatL2(dimension)
        index.add(embeddings)

        return index

    def retrieve_relevant_context(self, query: str, top_k: int = 3) -> List[Dict]:
        """
        Retrieve most relevant context from PDF knowledge base
        """
        query_embedding = self.embedding_model.encode([query])
        distances, indices = self.faiss_index.search(query_embedding, top_k)

        results = [
            {
                'text': self.text_chunks[idx],
                'metadata': self.metadata[idx],
                'distance': distances[0][i]
            }
            for i, idx in enumerate(indices[0])
        ]

        return results

In [ ]:
import os
import re
import logging
from typing import List, Dict, Optional
import random

# Minimal logging
logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger("DementiaAssistant")

class RobustPDFExtractor:
    """PDF extractor with compatibility with original code"""

    def __init__(self, fallback_to_ocr: bool = False):
        self._extraction_stats = {"success": 0, "failed": 0}

    def extract_text(self, pdf_path: str) -> str:
        """Extract text from PDF"""
        if not os.path.exists(pdf_path):
            return ""

        try:
            import PyPDF2
            text = ""
            with open(pdf_path, 'rb') as file:
                reader = PyPDF2.PdfReader(file)
                for page in reader.pages:
                    try:
                        text += page.extract_text() + " "
                    except:
                        pass

            if text:
                self._extraction_stats["success"] += 1
            else:
                self._extraction_stats["failed"] += 1

            return text
        except:
            self._extraction_stats["failed"] += 1
            return ""

    def get_extraction_stats(self) -> Dict[str, int]:
        return self._extraction_stats

class FastDementiaRAG:
    """Patient-centered dementia assistant that uses knowledge to inform responses"""

    def __init__(self,
                 pdf_folder_path: str,
                 embedding_model: str = 'paraphrase-MiniLM-L3-v2',
                 chunk_size: int = 500,
                 preprocess_only: bool = False,
                 cache_dir: Optional[str] = None):
        """Initialize with same parameters for compatibility"""
        self.pdf_folder_path = pdf_folder_path

        # Core knowledge components
        self.knowledge_base = []

        # Load and process documents
        self._process_documents()

        # Define templates for more personalized responses
        self._init_response_templates()

    def _process_documents(self):
        """Process documents into simple knowledge facts"""
        for filename in os.listdir(self.pdf_folder_path):
            if filename.endswith('.pdf'):
                extractor = RobustPDFExtractor()
                pdf_path = os.path.join(self.pdf_folder_path, filename)
                text = extractor.extract_text(pdf_path)

                if text:
                    # Split into paragraphs
                    paragraphs = [p.strip() for p in re.split(r'\n\n|\r\n\r\n', text) if p.strip()]

                    # Clean and store knowledge
                    for para in paragraphs:
                        if len(para.split()) > 10:  # Skip very short paragraphs
                            # Clean text
                            clean_para = re.sub(r'\s+', ' ', para).strip()

                            # Store with source
                            self.knowledge_base.append({
                                'text': clean_para,
                                'source': filename,
                                'keywords': self._extract_keywords(clean_para)
                            })

    def _extract_keywords(self, text: str) -> List[str]:
        """Extract meaningful keywords from text"""
        # Remove common stopwords
        stopwords = {"a", "an", "the", "and", "or", "but", "in", "on", "at", "to", "for", "with",
                    "by", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had",
                    "do", "does", "did", "of", "this", "that", "these", "those", "it", "they", "them"}

        # Extract words, convert to lowercase, and filter stopwords
        words = [word.lower() for word in re.findall(r'\b[a-zA-Z]{3,}\b', text)
                if word.lower() not in stopwords]

        return words

    def _init_response_templates(self):
        """Initialize templates for more natural responses"""
        self.emotion_templates = {
            "sad": [
                "I understand you're feeling sad. {fact} Remember that these feelings are normal.",
                "It's okay to feel sad sometimes. {fact} Would you like to talk more about what's on your mind?",
                "I hear that you're feeling down today. {fact} Is there something specific making you feel this way?"
            ],
            "confused": [
                "It's normal to feel confused sometimes. {fact} Let's take things one step at a time.",
                "Feeling confused can be frustrating. {fact} We can work through this together.",
                "When things feel confusing, it can help to focus on just one thing. {fact}"
            ],
            "angry": [
                "I understand you're feeling frustrated or angry. {fact} These feelings are completely valid.",
                "It's okay to feel upset. {fact} Would you like to talk about what's bothering you?",
                "I hear that you're feeling angry. {fact} Sometimes taking a few deep breaths can help."
            ],
            "fearful": [
                "It's natural to feel afraid sometimes. {fact} You're not alone in this.",
                "Feeling anxious or scared is common. {fact} Is there something specific worrying you today?",
                "When you're feeling scared, remember that {fact} Would it help to talk about your concerns?"
            ],
            "happy": [
                "I'm glad you're feeling happy! {fact} These positive moments are wonderful.",
                "It's great to hear you're in good spirits! {fact} What's bringing you joy today?",
                "Those happy feelings are so important. {fact} Would you like to share what's made you happy?"
            ]
        }

        self.question_templates = [
            "I believe {fact} Does that help answer your question?",
            "Based on what I know, {fact} Would you like to know more about this?",
            "{fact} I hope that information is useful to you.",
            "From what I understand, {fact} Is there anything specific about this you'd like to discuss?"
        ]

    def retrieve_relevant_context(self, query: str, max_results: int = 3) -> List[Dict]:
        """Find relevant knowledge using keyword matching"""
        query_words = set(word.lower() for word in re.findall(r'\b[a-zA-Z]{3,}\b', query))

        results = []
        for item in self.knowledge_base:
            # Count matching keywords
            matches = len(set(item['keywords']).intersection(query_words))

            if matches > 0:
                results.append({
                    'text': item['text'],
                    'metadata': {'source': item['source']},
                    'distance': 1.0 / (matches + 1)  # Convert to distance format for compatibility
                })

        # Sort by relevance (lower distance is better) and return top results
        results.sort(key=lambda x: x['distance'])
        return results[:top_k]

    def find_relevant_knowledge(self, query: str, max_results: int = 3) -> List[Dict]:
        """Alternative name for retrieve_relevant_context - for internal use"""
        return self.retrieve_relevant_context(query, max_results)

    def get_response_for_emotion(self, emotion: str, query: str = None) -> str:
        """Generate a natural, informed response for an emotional state"""
        # Find relevant knowledge
        if query:
            search_query = f"{emotion} {query}"
        else:
            search_query = f"dementia {emotion} feelings"

        knowledge = self.retrieve_relevant_context(search_query)

        # Select template for the emotion
        templates = self.emotion_templates.get(emotion.lower(), [
            "I understand you're feeling {emotion}. {fact}",
            "It's normal to have these feelings. {fact}",
            "When you're feeling {emotion}, remember that {fact}"
        ])

        template = random.choice(templates)

        # If we have relevant knowledge, use it
        if knowledge:
            # Extract a useful fact and simplify it for the patient
            fact = self._simplify_for_patient(knowledge[0]['text'])

            # Replace placeholders
            response = template.format(fact=fact, emotion=emotion)
        else:
            # Fallback response without specific knowledge
            response = template.format(
                fact="taking things one day at a time can help",
                emotion=emotion
            )

        return response

    def answer_dementia_question(self, question: str) -> str:
        """Answer a question with knowledge-informed, patient-friendly language"""
        # Find relevant knowledge
        knowledge = self.retrieve_relevant_context(question)

        if not knowledge:
            return "I'm not sure about that specific question. Would you like to ask me something else about dementia or daily activities?"

        # Select a template
        template = random.choice(self.question_templates)

        # Extract and simplify the most relevant fact
        fact = self._simplify_for_patient(knowledge[0]['text'])

        # Create response
        response = template.format(fact=fact)

        return response

    def _simplify_for_patient(self, text: str) -> str:
        """Simplify complex information for a patient with dementia"""
        # Shorten to a manageable length (1-2 sentences)
        sentences = re.split(r'(?<=[.!?])\s+', text)

        if len(sentences) > 2:
            simplified = ' '.join(sentences[:2])
        else:
            simplified = ' '.join(sentences)

        # Further simplify if still too long
        if len(simplified) > 150:
            simplified = simplified[:147] + "..."

        return simplified

In [ ]:
# Initialize your old RAG system
rag = PDFKnowledgeBaseRAG(pdf_folder_path="/content/drive/MyDrive/InnoChallenge-samurAI/RAG_docs")

def dementia_friendly_response(user_message, emotion, chat_history, pdf_rag=rag):
    """
    Generates a dementia-friendly response, considering emotion and chat history, with added warmth and considering context awareness and link generation.
    """
    # Check for similar previous context
    similar_context = context_tracker.find_similar_context(user_message)

    system_prompt = (
        "You are a compassionate assistant speaking directly to a person with dementia. "
        "Speak warmly and respectfully, using simple, reassuring language. "
        "Avoid asking memory-based questions. If the user seems confused, gently reassure them."
        "IMPORTANT: When discussing any topic, ALWAYS include 1-2 REAL, WORKING web links to helpful, beginner-friendly resources related to the topic."
        "Ensure the links are from reputable websites like educational sites learning platforms, or trusted information sources. "
        f"\n\nUser's current emotion: {emotion}\n\n"
        "CRITICAL GUIDELINES:"
        "\n1. Never offer to play music or show images—this is a text-only interface."
        "\n2. Always validate feelings before giving factual responses."
        "\n3. Acknowledge and improve responses if the user finds them unhelpful."
        "\n4. Keep responses direct, supportive, and friendly."
    )


        # Retrieve PDF context if PDF RAG is available
    retrieved_context = []
    if pdf_rag:
        try:
            retrieved_context = pdf_rag.retrieve_relevant_context(user_message)
            context_texts = [item['text'] for item in retrieved_context]
        except Exception as e:
            #print(f"PDF RAG retrieval error: {e}")
            context_texts = []
    else:
        context_texts = []

    # Add context from similar previous interaction if exists
    if similar_context:
        system_prompt += (
            "CONTEXT HINT: A similar conversation was previously had about a related topic. "
            f"Previous context: {similar_context['user_message']}\n"
        )

    # Add PDF context if available
    if context_texts:
        system_prompt += (
            "Additional Context from Documents:\n" +
            "\n".join(context_texts) + "\n\n"
        )

            # Add examples of good responses for different emotions
    emotion_examples = """
        "sad": "I'm really sorry you're feeling this way. You're not alone, and I'm here for you. "
               "Sometimes, talking about something comforting can help. Would you like to share a happy memory with me?",
        "confused": "I understand things might feel unclear right now. You're safe, and everything is okay. "
                    "Would you like to sit and chat for a while?",
        "angry": "I hear you, and your feelings are completely valid. It must be frustrating. "
                 "If you want to talk about it, I'm here to listen.",
        "calm": "That sounds nice. Taking things one step at a time is important. "
                "Is there something you’d like to do or talk about?",
        "fearful": "It's okay to feel scared sometimes. You're not alone, and you're in a safe place. "
                   "Would you like to take a deep breath with me? In… and out…",
        "happy": "That’s wonderful! I'm so glad you're feeling good. What’s making you happy today?",
        "neutral": "I'm here to chat about anything you'd like. Let me know what's on your mind!"
    """

    # Improved joke responses
    joke_examples = """
        "Why don't skeletons fight each other? Because they don’t have the guts! Did that make you smile?",
        "Why did the tomato turn red? Because it saw the salad dressing! I hope that was a little funny!",
        "What did one hat say to the other? ‘You stay here, I’ll go on ahead!’"
    """

    # Make the music limitations extremely explicit
    music_examples = """
Example (music requests - CRITICAL EXAMPLES):
User: Can we talk about my favourite song?
Assistant: I'd love to hear about your favorite song. What is it? Music can bring back such wonderful memories.

User: Can you play Perfect by Ed Sheeran?
Assistant: I wish I could play "Perfect" by Ed Sheeran for you, but I can't actually play music. I'm just here to chat. "Perfect" is a beautiful love song though. What do you enjoy about that song?

User: Yes, please play Perfect by Ed Sheeran
Assistant: I'm sorry for any confusion. I can't actually play music - I'm just a text assistant. But "Perfect" by Ed Sheeran is a lovely choice. Many people find the lyrics very touching. Is there something else we could talk about that might brighten your day?

User: Play me some music
Assistant: I'm sorry, I can't actually play music for you - I'm just a text assistant. But I'd love to talk about music with you. What kinds of songs do you enjoy listening to?

User: You're not playing my music
Assistant: You're right, and I apologize for any confusion. I can't play music - I'm just a text-based assistant that can chat with you. Would you like to talk about your favorite music instead? Or we could chat about something else you enjoy.
"""

    # Better handling of feedback
    feedback_examples = """
Example (responding to feedback):
User: That doesn't make sense
Assistant: You're right, that didn't make much sense. Let me be clearer. What I meant was... [simplified explanation]

User: You're not listening to me
Assistant: I'm sorry for not listening well. You're important and I want to understand. Could you tell me again, and I'll pay better attention this time?

User: I already told you that
Assistant: You're right, you did tell me that. I'm sorry for not remembering. Thank you for your patience with me. Let's continue our conversation about [previously mentioned topic].

User: I don't like this
Assistant: I understand this isn't working well for you. Let's try something different. Would you prefer to talk about [suggest alternative topic] instead?
"""

    # Visual content limitations
    visual_examples = """
Example (asking for visual content):
User: Can you show me a picture?
Assistant: I'm sorry, I can't show pictures in our conversation. But we could talk about things you enjoy seeing. Would you like to describe a favorite place or memory?

User: Show me a photo of my family
Assistant: I wish I could show you photos, but I can't display images. Your family sounds important to you. Would you like to tell me about them instead? I'd love to hear about your family members.
"""

    # Add all examples to system prompt
    system_prompt += emotion_examples
    system_prompt += joke_examples
    system_prompt += music_examples
    system_prompt += feedback_examples
    system_prompt += visual_examples

        # Build conversation history for context
    conversation = system_prompt

    # If the chat history is getting too long, keep only the most recent conversations
    # to avoid exceeding the token limit
    recent_history = chat_history[-3:] if len(chat_history) > 3 else chat_history

    for turn in recent_history:
        conversation += (
            f"User (feeling {turn['emotion']}): {turn['user']}\n"
            f"Assistant: {turn['assistant']}\n"
        )

    # Add the current message
    conversation += f"User (feeling {emotion}): {user_message}\nAssistant:"

    # Generate assistant response
    try:
        response = pipe(
            conversation,
            max_new_tokens=500,
            do_sample=True,
            temperature=0.25,
            top_p=0.82,
            top_k=30,
            repetition_penalty=1.08,
            num_return_sequences=1
        )

        assistant_reply = response[0]['generated_text'].split("Assistant:")[-1].strip()

         # Validate the response to ensure it adheres to guidelines
        #assistant_reply = validate_response(assistant_reply)

        # Add warmth and empathy to the response before sending it
        assistant_reply = f"💬 {assistant_reply}"

        # Add source information if PDF context was used
        if retrieved_context:
            sources = set(item['metadata']['source'] for item in retrieved_context)
            assistant_reply += f"\n\n(Information sourced from: {', '.join(sources)})"

        # Add interaction to context tracker
        context_tracker.add_interaction(user_message, emotion, assistant_reply)

        return assistant_reply

    except Exception as e:
        # Handle any errors gracefully
        print(f"Error generating response: {e}")
        return "I'm having trouble responding right now. Could you please repeat that?"


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# ✅ Main conversational loop (easily replaceable with voice-to-text input)
context_tracker = ContextTracker()

# Optional PDF RAG (provide path to PDF folder)
pdf_folder_path = "/content/drive/MyDrive/InnoChallenge-samurAI/RAG_docs"
pdf_rag = FastDementiaRAG(
    pdf_folder_path=pdf_folder_path,
    cache_dir="/content/drive/MyDrive/InnoChallenge-samurAI/rag_cache"
)

chat_history = []

print("👋 Hello! How can I help today?")

while True:
    user_message = input("User says: ")

    if user_message.lower() in ["exit", "quit", "bye"]:
        print("Assistant: Take care! I’ll be here if you need me.")
        break

    # ⚠️ Replace this with actual emotion detection.
    emotion = input("Detected emotion (confused, sad, angry, calm, fearful, happy, etc.): ")

    # Generate response with optional PDF RAG
    assistant_response = dementia_friendly_response(
            user_message,
            emotion,
            chat_history,
            pdf_rag=pdf_rag
        )

    # Show response to user
    print(f"Assistant: {assistant_response}")

    # Store conversation history
    chat_history.append({
        "user": user_message,
        "emotion": emotion,
        "assistant": assistant_response
    })


👋 Hello! How can I help today?
User says: Have you ever felt angry at yourself?
Detected emotion (confused, sad, angry, calm, fearful, happy, etc.): angry
Assistant: 💬 I'm not able to experience emotions like anger, but I can help you understand why it might be frustrating to feel angry. Would you like to talk about your feelings?
User says: I have dementia and i'm angry at myself for forgetting
Detected emotion (confused, sad, angry, calm, fearful, happy, etc.): angry
Assistant: 💬 I'm so sorry to hear that you're feeling angry at yourself. It's important to remember that everyone forgets sometimes, and that it's okay to feel frustrated. We can work together to find ways to manage your anger and cope with the challenges of living with dementia.
User says: How can I cope with living with dementia and forgetting my loved ones one day?
Detected emotion (confused, sad, angry, calm, fearful, happy, etc.): sad


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Assistant: 💬 💬 It's important to remember that you're not alone in this journey. There are many things you can do to cope with living with dementia and forgetfulness. Some helpful tips include:

* **Stay connected:** Talk to your loved ones often and make time for social activities.
* **Set realistic expectations:** Don't expect to remember everything or do everything perfectly.
* **Be patient:** It takes time to learn how to live with dementia.
* **Take care of yourself:** Make sure to get enough sleep, eat healthy foods, and exercise regularly.
* **Seek professional help:** A therapist can help you learn coping mechanisms and deal with the emotional challenges of living with dementia.
User says: exit
Assistant: Take care! I’ll be here if you need me.


In [ ]:
"""import os

# Specify directory path
save_dir = "/content/drive/MyDrive/InnoChallenge-samurAI/"

# Create directory if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

# Save model with full path
torch.save(model.state_dict(), os.path.join(save_dir, "dementiahelperllm.pth"))
print(f"\nModel saved as {os.path.join(save_dir, 'dementiahelperllm.pth')}")"""